# Model 2 — Hierarchical DDM
**Outputs:**
- `models/results/hierarchical_fit.pkl` — serialised CmdStanPy fit object
- `models/results/hierarchical_summary.csv` — posterior summary for all parameters

## Imports and config

In [2]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from cmdstanpy import CmdStanModel

DATA_PATH = Path("../EDA/data/dataset.csv")
STAN_FILE = Path("hierarchical.stan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# MCMC settings
N_CHAINS  = 4
N_WARMUP  = 1000
N_SAMPLES = 1000
SEED = 42

assert DATA_PATH.exists(), f"dataset.csv not found at {DATA_PATH}"
assert STAN_FILE.exists(), f"Stan file not found at {STAN_FILE}"

print(f"Stan file : {STAN_FILE}")
print(f"Data path : {DATA_PATH}")
print(f"Chains    : {N_CHAINS} x {N_WARMUP} warmup + {N_SAMPLES} sampling")

Stan file : hierarchical.stan
Data path : ../EDA/data/dataset.csv
Chains    : 4 x 1000 warmup + 1000 sampling


## Load and prepare data

In [3]:
def load_and_validate(path: Path) -> pd.DataFrame:
    """Load dataset.csv and validate required columns are present."""
    df = pd.read_csv(path)
    required = ["player_id", "choice", "RT", "difficulty_bin", "distance"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    print(f"Loaded: {len(df):,} trials, {df['player_id'].nunique()} players")
    print(f"RT range: {df['RT'].min():.3f}s -- {df['RT'].max():.3f}s")
    print(f"Choice balance: {df['choice'].mean():.1%} offensive")
    return df


def build_stan_data(df: pd.DataFrame) -> dict:
    """
    Build the data dict for Stan.
    """
    return {
         "N":          len(df),
        "J":          df["player_id"].nunique(),
        "player":     (df["player_id"] + 1).astype(int).tolist(),
        "choice":     df["choice"].astype(int).tolist(),
        "RT":         df["RT"].tolist(),
        "difficulty": df["difficulty_bin"].astype(int).tolist()
    }


df = load_and_validate(DATA_PATH)
stan_data = build_stan_data(df)

print("\nStan data summary:")
for k, v in stan_data.items():
    if isinstance(v, list):
        print(f"  {k}: list of {len(v)}")
    else:
        print(f"  {k}: {v}")

Loaded: 22,263 trials, 22 players
RT range: 0.100s -- 2.933s
Choice balance: 60.3% offensive

Stan data summary:
  N: 22263
  J: 22
  player: list of 22263
  choice: list of 22263
  RT: list of 22263
  difficulty: list of 22263


## Compile Stan model

In [4]:
model = CmdStanModel(stan_file=str(STAN_FILE))
print("Model compiled successfully")

Model compiled successfully


## Run MCMC

4 chains x 1000 warmup + 1000 sampling

In [ ]:
fit = model.sample(
    data=stan_data,
    chains=N_CHAINS,
    iter_warmup=N_WARMUP,
    iter_sampling=N_SAMPLES,
    seed=SEED,
    show_progress=True
)

print("Sampling complete")
print(f"Total post-warmup draws: {N_CHAINS * N_SAMPLES:,}")

16:42:22 - cmdstanpy - INFO - CmdStan start processing
chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/2000 [00:00<22:28,  1.48it/s, (Warmup)]


chain 4:   0%|          | 1/2000 [00:00<22:40,  1.47it/s, (Warmup)]

chain 3:   0%|          | 1/2000 [00:00<22:54,  1.45it/s, (Warmup)]

## Save fit

In [10]:
def save_fit(fit, results_dir: Path) -> None:
    """Save the CmdStanPy fit object and posterior summary"""
    fit_path     = results_dir / "hierarchical_fit.pkl"
    summary_path = results_dir / "hierarchical_summary.csv"

    with open(fit_path, "wb") as f:
        pickle.dump(fit, f)

    summary = fit.summary()
    summary.to_csv(summary_path)

    print(f"Saved fit : {fit_path}")
    print(f"Saved summary : {summary_path}")


save_fit(fit, RESULTS_DIR)

Saved fit : results/hierarchical_fit.pkl
Saved summary : results/hierarchical_summary.csv


In [12]:
print(df["RT"].describe())
print(f"Min RT: {df['RT'].min():.4f}")
print(f"Mean RT: {df['RT'].mean():.4f}")

count    22263.000000
mean         0.822987
std          0.320577
min          0.100000
25%          0.633333
50%          0.800000
75%          0.966667
max          2.933333
Name: RT, dtype: float64
Min RT: 0.1000
Mean RT: 0.8230


## Convergence diagnostics

Requirements:
- R-hat < 1.01 for ALL parameters including hyperparameters
- ESS > 400 for ALL parameters
- Zero divergent transitions

In [11]:
def check_convergence(fit) -> pd.DataFrame:
    """
    Check R-hat and ESS for all parameters.
    Flags any parameter that fails the threshold.
    Returns the full summary dataframe.
    """
    summary  = fit.summary()
    diagnose = fit.diagnose()

    # Divergences
    if "0 of" in diagnose:
        print("No divergent transitions")
    else:
        print("Divergent transitions detected")
        print(diagnose)

    # R-hat
    rhat_col = [c for c in summary.columns if "r_hat" in c.lower()][0]
    bad_rhat = summary[summary[rhat_col] >= 1.01]
    if bad_rhat.empty:
        print(f"All R-hat < 1.01 ({len(summary)} parameters checked)")
    else:
        print(f"{len(bad_rhat)} parameters with R-hat >= 1.01:")
        print(bad_rhat[[rhat_col]].to_string())

    # ESS
    ess_col = [c for c in summary.columns if "ess_bulk" in c.lower()][0]
    bad_ess = summary[summary[ess_col] < 400]
    if bad_ess.empty:
        print("All ESS > 400")
    else:
        print(f"{len(bad_ess)} parameters with ESS < 400:")
        print(bad_ess[[ess_col]].to_string())

    return summary


summary = check_convergence(fit)

Divergent transitions detected
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
7993 of 8000 (99.91%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.00, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

The following parameters had fewer than 0.001 effective draws per transition:
  mu_v_easy, sigma_v_easy, mu_v_hard, sigma_v_hard, mu_beta_logit, sigma_beta, mu_log_a, sigma_a, mu_log_t, sigma_t, z_v_easy[1], z_v_easy[3], z_v_easy[4], z_v_easy[5], z_v_easy[6], z_v_easy[8], z_v_easy[10], z_v_easy[11], z_v_easy[12], z_v_eas

## Key hyperparameter posteriors

In [ ]:
def print_hyperparameter_summary(summary: pd.DataFrame) -> None:
    """Print posterior summary for key group-level hyperparameters."""
    key_params = [
        "mu_v_easy", "mu_v_hard", "sigma_v_easy", "sigma_v_hard",
        "mu_log_a", "mu_log_t", "mu_beta_logit",
        "mu_v_diff", "p_v_easy_gt_v_hard", "group_beta",
    ]
    available = [p for p in key_params if p in summary.index]
    print("Group-level hyperparameter posteriors:")
    print(summary.loc[available].to_string())


print_hyperparameter_summary(summary)